# Demo -- end-to-end GNN-BERT context inference

One track, start to finish:

    waveform -> log-mel / chroma / MFCC -> segments -> graph
             +  caption -> BERT
             -> cross-attention fusion
             -> tags, valence/arousal, and the attended caption tokens

Prerequisites, for whichever corpus you trained on:

```bash
python src/preprocess.py --config configs/mtat.yaml
python src/train.py task3 --config configs/mtat.yaml
```

Set `CONFIG` in the next cell to the same preset you trained with. The
checkpoint records the corpus it was trained on, so a mismatch between the
weights and the processed data is reported rather than crashing on a tensor
shape.

In [ ]:
import sys, json
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import torch
import matplotlib.pyplot as plt

from utils import load_config, load_json, get_device, resolve

CONFIG = "configs/mtat.yaml"      # <-- the preset you trained with, or None for config.yaml
RESULTS = None                    # <-- override the results dir, or None to take it from the config

cfg = load_config(REPO / CONFIG if CONFIG else REPO / "config.yaml")
device = get_device(cfg.get("device", "auto"))

processed = resolve(cfg, "processed")
splits = resolve(cfg, "splits")
results = Path(RESULTS) if RESULTS else resolve(cfg, "results")

print("repo     :", REPO)
print("device   :", device)
print("corpus   :", cfg["dataset"]["source"], "| graph:", cfg["graph"]["kind"],
      "| segment:", cfg["audio"]["segment_seconds"], "s")
print("processed:", processed)
print("results  :", results)

## 1. Take an audio clip

By default this picks a real clip from the **test** split, so the demo shows
inference on data the model never saw. Set `AUDIO_PATH` to run on any file of
your own, or `USE_SYNTHETIC = True` to compose one.

In [ ]:
from audio_features import load_audio, extract_track_features
from data.synthetic import synthesize_track, build_caption, build_tags

AUDIO_PATH = None        # e.g. "../data/raw/gtzan/genres_original/jazz/jazz.00000.wav"
USE_SYNTHETIC = False

index = {r["track_id"]: r for r in load_json(processed / "index.json")}
test_ids = load_json(splits / "test.json")

record, truth = None, {}
if AUDIO_PATH:
    y = load_audio(AUDIO_PATH, cfg["audio"]["sample_rate"], cfg["dataset"]["clip_seconds"])
    caption = "a music track"
elif USE_SYNTHETIC:
    y, truth = synthesize_track("jazz", "melancholic", seed=1234,
                                duration=cfg["dataset"]["clip_seconds"])
    caption = build_caption(truth, np.random.default_rng(0), label_dropout=1.0)
else:
    # First test clip whose source audio is still on disk.
    record = next((index[t] for t in test_ids
                   if index.get(t, {}).get("audio_path")
                   and Path(index[t]["audio_path"]).exists()), None)
    if record is None:
        raise SystemExit("no test clip has its audio on disk -- set AUDIO_PATH "
                         "or USE_SYNTHETIC = True")
    y = load_audio(record["audio_path"], cfg["audio"]["sample_rate"],
                   cfg["dataset"]["clip_seconds"])
    caption = record["text"]

print("samples :", y.shape[0])
print("caption :", caption)
if record:
    print("track   :", record["track_id"], "| artist:", record["artist"])
    print("true tags:", record["tags"])
elif truth:
    print("true tags:", build_tags(truth))

## 2. Features and graph

In [ ]:
from graph_builder import build_graph, graph_summary

tf = extract_track_features(y, "demo_track", cfg)
graph = build_graph(tf, cfg)
print(json.dumps(graph_summary(graph), indent=2))
print("segment times:", tf.segment_times())

## 3. Tokenise the caption and assemble one batch

In [ ]:
from transformers import AutoTokenizer
from torch_geometric.data import Batch

tokenizer = AutoTokenizer.from_pretrained(cfg["text"]["model_name"])
enc = tokenizer([caption], padding="max_length", truncation=True,
                max_length=cfg["text"]["max_length"], return_tensors="pt")

tags = load_json(processed / "label_space.json")["tags"]

graph.input_ids = enc["input_ids"]
graph.attention_mask = enc["attention_mask"]
graph.y = torch.zeros(1, len(tags))
graph.va = torch.zeros(1, 2)
graph.va_mask = torch.zeros(1)
graph.text = caption

batch = Batch.from_data_list([graph]).to(device)
batch

## 4. Load the trained fusion model

`load_checkpoint` compares the checkpoint's recorded corpus and label-space
size against the processed data and raises `CheckpointMismatch` with an
explanation if they disagree — rather than failing inside `load_state_dict`
with a bare tensor-shape error.

In [ ]:
from fusion_model import build_fusion_model, top_attended_tokens
from engine import emotion_stats, load_checkpoint
from data.dataset import MusicContextDataset

ckpt_path = results / "checkpoints" / "task3_fusion_cross_attention.pt"
assert ckpt_path.exists(), (
    f"no checkpoint at {ckpt_path}\n"
    f"train it first:  python src/train.py task3 --config {CONFIG}")

train_ds = MusicContextDataset(processed, splits, "train")
predict_emotion = bool(train_ds.meta.get("has_emotion"))

blob = load_checkpoint(ckpt_path, train_ds, device)     # guards the label space
print("checkpoint provenance:", json.dumps(blob.get("provenance"), indent=2))

model = build_fusion_model(cfg, train_ds.node_feature_dim,
                           len(tags), predict_emotion).to(device)
model.load_state_dict(blob["state_dict"])
model.eval()

with torch.no_grad():
    out = model(batch)
    probs = torch.sigmoid(out["tag_logits"])[0].cpu().numpy()
print("\nloaded:", ckpt_path.name, "| best epoch:", blob.get("best_epoch"))

## 5. Predictions\n\nThe threshold is the one tuned on the validation split during training -- not a flat 0.5, which is a poor operating point for sparse multi-label targets.

In [ ]:
metrics = load_json(results / "metrics.json")
threshold = metrics["task3_fusion_cross_attention"]["threshold"]
order = np.argsort(-probs)[:10]

print(f"threshold tuned on val = {threshold:.2f}\n")
print(f"{'tag':<18}{'score':>8}   predicted")
for i in order:
    print(f"{tags[i]:<18}{probs[i]:>8.4f}   {'YES' if probs[i] >= threshold else ''}")

if predict_emotion and out["emotion"] is not None:
    mean, std = emotion_stats(train_ds)
    va = out["emotion"][0].cpu() * std + mean
    print(f"\nvalence {va[0]:.2f} / arousal {va[1]:.2f}  (DEAM 1-9 scale)")

## 6. What the model attended to\n\nThe graph vector queries the caption tokens, so the attention weights say which words the audio structure aligned with.

In [ ]:
attended = top_attended_tokens(tokenizer, batch.input_ids, out["attention"], k=8)
if attended:
    for token, weight in attended[0]:
        bar = "#" * int(weight * 200)
        print(f"{token:<14}{weight:.4f}  {bar}")
else:
    print("no attention (fusion.mode is not cross_attention)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
top = order[:10]
ax.barh([tags[i] for i in top], probs[top])
ax.axvline(threshold, color="crimson", ls="--", label=f"threshold {threshold:.2f}")
ax.invert_yaxis(); ax.set_xlabel("probability"); ax.legend()
ax.set_title("predicted context tags")
plt.tight_layout()